# 02 — The soft margin and the cost C

*Notebook 2 of 5 · Support Vector Machines*

In the last notebook the two classes were cleanly separable, so a widest street existed. Real data is
rarely that tidy: here the two penguin species overlap by one stubborn point, and **no** street
separates them. The fix is to allow a few **violations** — let points sit inside the street, or even on
the wrong side — and charge a cost for each. One dial, **C**, sets how steep that charge is, and the
penalty a violation pays turns out to be the **hinge loss**, a close cousin of Chapter 03's log-loss.

**Prerequisites**
- NB 1 — the maximum margin and its support vectors; the margin width is $2/\lVert w \rVert$.
- Chapter 03, NB 3 — log-loss (the training objective there); we line the hinge loss up against it.
- Chapter 01, NB 2 — the scale trap (named here, paid off in NB 5).

**What you'll be able to do**
- Explain why real, overlapping data needs a *soft* margin.
- Define **slack** and the **hinge loss**, and compute them by hand.
- Say what the cost **C** trades, and predict the effect of turning it up or down.
- Read the margin width and the support-vector count off a **C**-sweep.
- Connect the hinge loss to Chapter 03's log-loss.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ml_course import colors, datasets, viz

viz.use_course_style()
SEED = 0
np.random.seed(SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

df = datasets.load_penguins()
features = ["bill_length_mm", "flipper_length_mm"]
X = StandardScaler().fit_transform(df[features].to_numpy())
y = (df["species"] == "Gentoo").astype(int).to_numpy()
y_pm = 2 * y - 1  # map {0, 1} -> {-1, +1}, the convention the hinge formula needs

hard = SVC(kernel="linear", C=1e6).fit(X, y)
print(f"penguins (standardized): n = {len(y)}, class counts (Adelie, Gentoo) = {np.bincount(y)}")
print(f"near-hard-margin SVM (C=1e6) train accuracy: {hard.score(X, y):.4f}")
print(f"-> not 1.000: {int((hard.predict(X) != y).sum())} point(s) cannot be placed cleanly")

## When no street separates the classes

NB 1's hard margin assumed the two classes could be perfectly separated — every point kept strictly
off the street. These two penguin species almost can be, but not quite: one Adélie sits among the
Gentoos. Demanding zero violations now has **no solution**. We need a rule that bends: keep the street
as wide as we can, but let a few points break it, for a price.

## Allow violations, at a cost

The idea is to let a point sit **inside** the street, or even on the **wrong side** — but make it
**pay**. The size of the payment grows with how far the point intrudes. That "how far" is called the
point's **slack**. A point safely outside the street pays nothing; one that barely crosses the edge
pays a little; one deep on the wrong side pays a lot.

In [ ]:
wrong_idx = int(np.flatnonzero(hard.predict(X) != y)[0])
species = "Gentoo" if y[wrong_idx] == 1 else "Adelie"
print(f"the point a hard margin cannot place: index {wrong_idx}, true class {species}")
print(f"  standardized (bill, flipper) = {X[wrong_idx].round(2)}")
print("no straight street separates the two species cleanly -> we must let this point violate it")

## Slack and the hinge loss

Measure a point by its **functional margin**

$$ m = y\,(w \cdot x + b), $$

with the label $y \in \{-1, +1\}$. Reading $m$:

- $m \ge 1$: safely **outside** the street, on the right side — no violation.
- $0 < m < 1$: **inside** the street, still correct.
- $m < 0$: on the **wrong** side.

The price the point pays is the **hinge loss**

$$ \text{hinge} = \max(0,\ 1 - m): $$

zero while it stays safely outside ($m \ge 1$), then rising in a straight line as it intrudes. That
rising amount *is* the slack.

In [ ]:
soft = SVC(kernel="linear", C=1.0).fit(X, y)
f = soft.decision_function(X)
m = y_pm * f                       # functional margin  m = y (w.x + b)
hinge = np.maximum(0.0, 1.0 - m)

clear_i = int(np.argmax(m))                              # well outside the street
in_street = np.flatnonzero((m > 0) & (m < 1))
in_street_i = int(in_street[np.argmin(np.abs(m[in_street] - 0.5))])  # inside, correct side
wrong_i = int(np.flatnonzero(m < 0)[0])                  # on the wrong side

for label, i in [("well outside the street", clear_i),
                 ("inside the street      ", in_street_i),
                 ("on the wrong side       ", wrong_i)]:
    print(f"  {label}: m = {m[i]:+.3f}  ->  hinge = max(0, 1 - m) = {hinge[i]:.3f}")
print(f"  paying slack (hinge > 0) = {int((m < 1 - 1e-9).sum())};  "
      f"support vectors (m <= 1, on or inside the street) = {soft.n_support_.sum()};  "
      f"misclassified = {int((m < 0).sum())}")

**Read the result.** The point well outside the street pays **nothing** — the hinge is exactly
zero. The point inside the street pays a **fraction** of 1, and the point on the wrong side pays **more
than 1**, growing the farther it sits from where it should be. Add up everyone's hinge and you have the
total slack the boundary has tolerated. The **support vectors** are the points sitting **on or inside**
the street (functional margin $m \le 1$): those inside it or on the wrong side pay slack, while those
resting **exactly on** the edge ($m = 1$) are support vectors that pay nothing. So there are always at
least as many support vectors as slack-payers — here **17** support vectors against **15** paying slack,
the other two sitting on the edge (exactly the two zero-slack support vectors NB 1 had).

## The same shape as log-loss

Chapter 03 trained logistic regression by **log-loss**, which also punished being confident and wrong.
It helps to draw both losses as a function of the margin $m = y\,(w \cdot x + b)$ and compare their
shapes.

In [ ]:
margins = np.linspace(-1.5, 3.0, 300)
hinge_curve = np.maximum(0.0, 1.0 - margins)
logloss_curve = np.log1p(np.exp(-margins))  # log(1 + e^-m), the per-example logistic loss

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(margins, hinge_curve, color=colors.COLORS["model"], linewidth=2.2,
        label="hinge:  max(0, 1 - m)")
ax.plot(margins, logloss_curve, color=colors.COLORS["highlight"], linewidth=2.2, linestyle="--",
        label="log-loss:  log(1 + e^-m)")
ax.axvline(1.0, color=colors.COLORS["muted"], linewidth=1.0, linestyle=":")
ax.set_xlabel("margin   m = y (w . x + b)")
ax.set_ylabel("loss paid")
ax.set_title("Two ways to punish 'confident and wrong'")
ax.legend(loc="upper right")
plt.show()

**Read the figure.** Both losses fall as the margin grows, and both punish confident mistakes
(large loss when $m$ is very negative). The difference is on the right: the **hinge hits exactly zero**
at $m = 1$ (the dotted line) — once a point is safely past the street's edge it costs nothing more,
which is what lets the SVM keep a *margin* rather than chase every point. **Log-loss never quite
reaches zero**: it always asks each point to be a little more confident. Same instinct, two shapes.

## The cost C is the dial

Put the two halves together. The soft-margin SVM minimizes

$$ \tfrac{1}{2}\lVert w \rVert^2 \;+\; C \sum_i \text{hinge}_i. $$

The first term **widens** the street (small $\lVert w \rVert$ = large margin, from NB 1); the second
**pays** for the violations. The cost **C** is the exchange rate between them:

- **small C** — violations are cheap, so the model keeps a **wide** street and tolerates points inside
  it (more bias);
- **large C** — violations are expensive, so the model **narrows** the street to avoid them, heading
  back toward the hard margin (more variance).

In [ ]:
print(f"{'C':>8}  {'margin':>7}  {'n_SV':>5}  {'slack':>7}  {'wrong':>6}  {'train':>7}  {'CV':>7}")
for C in (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0):
    clf = SVC(kernel="linear", C=C).fit(X, y)
    mc = y_pm * clf.decision_function(X)
    margin = 2.0 / np.linalg.norm(clf.coef_[0])
    cv = cross_val_score(SVC(kernel="linear", C=C), X, y, cv=CV).mean()
    print(f"{C:8.2f}  {margin:7.3f}  {clf.n_support_.sum():5d}  {int((mc < 1 - 1e-9).sum()):8d}  "
          f"{int((mc < 0).sum()):6d}  {clf.score(X, y):7.4f}  {cv:7.4f}")

**Read the result.** As C rises the street **narrows** (margin 2.28 → 0.35) and the model leans
on **fewer** support vectors (124 → 6). Yet the accuracy barely moves (train and CV both stay around
0.99). On this near-separable data, **C is choosing the geometry** — how wide the street is and which
points hold it — not the score. (At large C the support-vector and slack columns part ways, e.g. 6
versus 3: the few remaining support vectors then rest exactly on the street's edge, paying no slack.)
Where C *also* moves accuracy is on harder, curved problems with a kernel; that is the next-but-one
notebook.

In [ ]:
fig, (ax_lo, ax_hi) = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, C in [(ax_lo, 0.1), (ax_hi, 100.0)]:
    clf = SVC(kernel="linear", C=C).fit(X, y)
    viz.plot_svm_decision(clf, X, y, ax=ax)
    margin = 2.0 / np.linalg.norm(clf.coef_[0])
    ax.set_title(f"C = {C}   (margin {margin:.2f}, {clf.n_support_.sum()} support vectors)")
    ax.set_xlabel("feature 1 (standardized)")
    ax.set_ylabel("feature 2 (standardized)")
fig.tight_layout()
plt.show()

**Read the figure.** Small C (left) keeps a **broad** street and lets many points sit inside it —
every point inside the street is a ringed support vector. Large C (right) **squeezes** the street until
almost nothing violates it, resting on a handful of points. Same data, two very different boundaries — chosen by a
single number.

In [ ]:
grid_C = np.logspace(-2, 3, 12)
margins_by_C, n_sv_by_C = [], []
for C in grid_C:
    clf = SVC(kernel="linear", C=C).fit(X, y)
    margins_by_C.append(2.0 / np.linalg.norm(clf.coef_[0]))
    n_sv_by_C.append(int(clf.n_support_.sum()))

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(grid_C, margins_by_C, color=colors.COLORS["model"], marker="o", linewidth=2,
         label="margin 2/||w||")
ax1.set_xscale("log")
ax1.set_xlabel("C (log scale)")
ax1.set_ylabel("margin width", color=colors.COLORS["model"])
ax2 = ax1.twinx()
ax2.plot(grid_C, n_sv_by_C, color=colors.COLORS["highlight"], marker="s", linewidth=2,
         linestyle="--", label="# support vectors")
ax2.set_ylabel("# support vectors", color=colors.COLORS["highlight"])
ax2.grid(False)
ax1.set_title("Raising C narrows the street and drops support vectors")
fig.tight_layout()
plt.show()

**Read the figure.** The two curves move together: a **wide** margin needs **many** support
vectors to hold it up, a **narrow** margin needs only a few. This is the bias–variance dial from NB 1's
complexity language, now in support-vector-machine form — and C is the knob that turns it.

## Honest limits

- On this near-separable data, **C mostly moves the geometry**, not the accuracy. Where C also moves
  accuracy is on harder, kernelized problems (NB 4) — we will tune it there for real.
- Everything here assumes the two features are on a **comparable scale** (we standardized them). *Why*
  an SVM is so sensitive to scale, and what happens when you forget, is the headline of NB 5.

## Your turn

1. **Easy — predict the street.** Without running anything, say whether a **small** or a **large** C
   gives the wider street, and explain why in one sentence.
2. **Medium — read the sweep.** From the C-sweep table, name the C that leans on the **fewest** support
   vectors, and say what that buys and what it risks.
3. **Harder — compute the hinge.** Three points have functional margins $m = 1.8$, $m = 0.3$, and
   $m = -0.5$. Compute each one's hinge loss, say which are support vectors, and which is misclassified.

## What you built

- You saw that real, overlapping data needs a **soft margin** — a hard margin had no solution here.
- You defined **slack** and the **hinge loss** $\max(0, 1-m)$ and computed them by hand: zero for a
  safely-classified point, a fraction inside the street, more than 1 on the wrong side.
- You connected the hinge to Chapter 03's **log-loss** — same instinct, the hinge with a flat zero past
  the margin.
- You read the cost **C** as the dial in $\tfrac{1}{2}\lVert w \rVert^2 + C\sum_i \text{hinge}_i$, and
  watched it trade margin width against violations: wide-and-forgiving ↔ narrow-and-strict.

**Vocabulary:** slack · violation · the hinge loss · functional margin · the cost C · soft margin.

## Going further (optional)

The soft margin is the optimization problem

$$ \min_{w,\,b,\,\xi}\ \tfrac{1}{2}\lVert w \rVert^2 + C\sum_i \xi_i
\quad\text{subject to}\quad y_i\,(w \cdot x_i + b) \ge 1 - \xi_i,\ \ \xi_i \ge 0, $$

where the optimal slack is exactly $\xi_i = \text{hinge}_i = \max(0, 1 - m_i)$. Like the hard margin,
its solution still depends on the data **only through dot products** between points — which is what the
next notebook exploits to bend the boundary (the **kernel trick**).

## References

- Cortes C, Vapnik V (1995). *Support-vector networks.* Machine Learning 20:273-297.
  DOI: [10.1007/BF00994018](https://doi.org/10.1007/BF00994018)
- Hastie T, Tibshirani R, Friedman J (2009). *The Elements of Statistical Learning*, §12.3 (the SVM as
  a penalized hinge loss). DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James G, Witten D, Hastie T, Tibshirani R (2021). *An Introduction to Statistical Learning*, §9.2.
  DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: 01 — The maximum margin · Next: 03 — The kernel trick*